# 05 – Salary model

Goal: predict the yearly salary of a posting (pooled model on all 35,910 LinkedIn salary rows, evaluated separately on data jobs).

1. Feature table (role, level, state, work type, remote, industry, company size, title text, skills).
2. Group train / test split by desc_group.
3. Baselines: global median, median by role + level + state.
4. CatBoost on log(salary).
5. Salary ranges (P10 / P50 / P90) and SHAP explanations.

## Step 1: Feature table

In [1]:
import re
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.skills import extract_skills

RAW = "../data/raw/linkedin"
sal = pd.read_parquet("../data/processed/linkedin_salaries.parquet")
print(sal.shape)

(35910, 38)


In [2]:
# Industry (first one per posting) and latest company size
ind = (pd.read_csv(f"{RAW}/jobs/job_industries.csv")
         .drop_duplicates("job_id")
         .merge(pd.read_csv(f"{RAW}/mappings/industries.csv"), on="industry_id", how="left"))

size = (pd.read_csv(f"{RAW}/companies/employee_counts.csv")
          .sort_values("time_recorded")
          .drop_duplicates("company_id", keep="last")[["company_id", "employee_count"]])

df = (sal.merge(ind[["job_id", "industry_name"]], on="job_id", how="left")
         .merge(size, on="company_id", how="left"))

print("rows:", len(df), "(must still be 35,910)")
print(df[["industry_name", "employee_count"]].isna().mean().round(3))

rows: 35910 (must still be 35,910)
industry_name     0.011
employee_count    0.014
dtype: float64


In [3]:
df["state"] = df["state"].fillna("NO_STATE")
df["remote"] = df["remote_allowed"].fillna(0).astype(int)
df["industry"] = df["industry_name"].fillna("Unknown")
df["log_company_size"] = np.log1p(df["employee_count"])
df["log_salary"] = np.log(df["salary"])

df["skills"] = df["description"].apply(extract_skills)   # ~2–3 minutes

skill_counts = df["skills"].explode().value_counts()
top_skills = skill_counts[skill_counts >= 0.01 * len(df)].index.tolist()
print(len(top_skills), "skills appear in at least 1% of postings")

for s in top_skills:
    df["skill_" + re.sub(r"\W", "_", s)] = df["skills"].apply(lambda x, s=s: s in x).astype(int)

19 skills appear in at least 1% of postings


In [4]:
CAT = ["role", "level", "state", "formatted_work_type", "industry"]
NUM = ["remote", "log_company_size"]
TEXT = ["title"]
SKILL_COLS = [c for c in df.columns if c.startswith("skill_")]

features = df[["job_id", "desc_group", "salary", "log_salary"] + CAT + NUM + TEXT + SKILL_COLS]
features.to_parquet("../data/processed/salary_features.parquet", index=False)

print(features.shape)
features.head(3)

(35910, 31)


,job_id,desc_group,salary,log_salary,role,level,state,formatted_work_type,industry,remote,...,skill_oracle,skill_linux,skill_r,skill_javascript,skill_power_bi,skill_git,skill_ci_cd,skill_tableau,skill_c__,skill_gcp
0,921716,35d1ac98bd02,38480.0,10.557894,Other,Unknown,NJ,Full-time,Real Estate,0,...,0,0,0,0,0,0,0,0,0,0
1,1829192,211dc20e5f21,83200.0,11.329003,Other,Unknown,CO,Full-time,Unknown,0,...,0,0,0,0,0,0,0,0,0,0
2,10998357,feeadef93690,55000.0,10.915088,Other,Lead/Principal,OH,Full-time,Restaurants,0,...,0,0,0,0,0,0,0,0,0,0


### Feature table

- 35,910 rows (no duplicates from the merges), 31 columns: role, level, state (NO_STATE for 17%), work type, industry (338 values, 1.1% unknown), log company size (1.4% missing), remote, title as text, 19 skill flags (skills in ≥ 1% of postings).
- Excluded on purpose: min / max / med salary, raw_mid, pay_period, compensation_type (built from the salary or only filled when a salary exists), views / applies / closed_time (known only after posting), company_name (phase 2).
- Saved as data/processed/salary_features.parquet.